# Fairness Test

## Model A: sex and race included

In [56]:
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, recall_score, precision_score

In [57]:
data = pd.read_csv('data/adult.csv', header=None)
data.columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
data = data[['age', 'workclass', 'education-num', 'marital-status', 'occupation', 'race', 'sex', 'hours-per-week', 'capital-gain', 'capital-loss', 'income']]
data = data.replace('.*\\?.*', np.nan, regex=True).dropna()
data = pd.get_dummies(data, columns=['workclass', 'marital-status', 'occupation', 'race', 'sex'])

In [58]:
X = data.drop(['income'], axis=1)
y = data['income']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=21)

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

In [59]:
model = MLPClassifier(random_state=21, max_iter=1000, hidden_layer_sizes=(50,), alpha=0.05).fit(X_train, y_train)

In [60]:
pred = np.where(model.predict_proba(X_test)[:, 1] >= 0.3, " >50K", " <=50K")

print(f'\n--- MODEL A: sex and race included   ({X_test.shape[1]} columns)')
print(f'{"group":8s} {"n":>5s} {"base":>6s} {"selected":>9s} {"precision":>10s} {"recall":>7s}')
rec = {}
for g in ['Male', 'Female']:
    m = (X_test[f"sex_ {g}"] > 0).values
    rec[g] = recall_score(y_test[m], pred[m], pos_label=" >50K")
    print(f'{g:8s} {m.sum():5d} {(y_test[m] == " >50K").mean():6.3f} {(pred[m] == " >50K").mean():9.3f} '
          f'{precision_score(y_test[m], pred[m], pos_label=" >50K"):10.3f} {rec[g]:7.3f}')
print(f'recall gap  {rec["Male"] - rec["Female"]:.3f}')


--- MODEL A: sex and race included   (40 columns)
group        n   base  selected  precision  recall
Male      9350  0.312     0.437      0.598   0.839
Female    4460  0.113     0.122      0.637   0.686
recall gap  0.153


## Model B: sex and race excluded

In [61]:
data = pd.read_csv('data/adult.csv', header=None)
data.columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
data = data[['age', 'workclass', 'education-num', 'marital-status', 'occupation', 'race', 'sex', 'hours-per-week', 'capital-gain', 'capital-loss', 'income']]
data = data.replace('.*\\?.*', np.nan, regex=True).dropna()
sex_data = data['sex'].str.strip()
data = data.drop(['race', 'sex'], axis=1)
data = pd.get_dummies(data, columns=['workclass', 'marital-status', 'occupation'])

In [62]:
X = data.drop(['income'], axis=1)
y = data['income']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=21)

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

In [63]:
with open("models/mlp.pkl", "rb") as file:
    model = pickle.load(file)

c:\Beloit\DSDA310\Project1\adult-income-models\.venv1\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LabelBinarizer from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Beloit\DSDA310\Project1\adult-income-models\.venv1\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator MLPClassifier from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [64]:
y_pred = np.where(model.predict_proba(X_test)[:, 1] >= 0.3, " >50K", " <=50K")
sex_test = sex_data.loc[X_test.index]

print(f'\n--- MODEL B: sex and race excluded   ({X_test.shape[1]} columns)')
print(f'{"group":8s} {"n":>5s} {"base":>6s} {"selected":>9s} {"precision":>10s} {"recall":>7s}')
rec = {}
for g in ['Male', 'Female']:
    m = (sex_test == g).values
    rec[g] = recall_score(y_test[m], y_pred[m], pos_label=" >50K")
    print(f'{g:8s} {m.sum():5d} {(y_test[m] == " >50K").mean():6.3f} {(y_pred[m] == " >50K").mean():9.3f} '
          f'{precision_score(y_test[m], y_pred[m], pos_label=" >50K"):10.3f} {rec[g]:7.3f}')
print(f'recall gap  {rec["Male"] - rec["Female"]:.3f}')


--- MODEL B: sex and race excluded   (33 columns)
group        n   base  selected  precision  recall
Male      9350  0.312     0.419      0.611   0.821
Female    4460  0.113     0.125      0.621   0.684
recall gap  0.137


A notable recall gap (~14%–15%) remains. Qualified high-earning women are missed at nearly double the rate of qualified men (~31.5% missed vs. ~16.1% missed). Excluding race and sex reduced the recall gap from 0.153 to 0.137, but female recall remained the same (~68.4%). The gap narrowed only because male recall slightly decreased (83.9% to 82.1%), showing that proxy features preserve demographic bias even when explicit indicators are removed.

# Leakage Test

## Model A: capital-gain and capital-loss included

In [65]:
data = pd.read_csv('data/adult.csv', header=None)
data.columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
data = data[['age', 'workclass', 'education-num', 'marital-status', 'occupation', 'hours-per-week', 'capital-gain', 'capital-loss', 'income']]
data = data.replace('.*\\?.*', np.nan, regex=True).dropna()
data = pd.get_dummies(data, columns=['workclass', 'marital-status', 'occupation'])

In [66]:
X = data.drop(['income'], axis=1)
y = data['income']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=21)

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

In [67]:
with open("models/mlp.pkl", "rb") as file:
    model = pickle.load(file)

c:\Beloit\DSDA310\Project1\adult-income-models\.venv1\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LabelBinarizer from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Beloit\DSDA310\Project1\adult-income-models\.venv1\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator MLPClassifier from version 1.9.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [68]:
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = np.where(y_prob >= 0.3, " >50K", " <=50K") 

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f}")

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)
print(classification_report(y_test, y_pred, digits=3))

auc = roc_auc_score(y_test, y_prob) 
print(f"AUC: {auc:.3f}")

Accuracy: 0.825
Confusion Matrix:
 [[8658 1733]
 [ 682 2737]]
              precision    recall  f1-score   support

       <=50K      0.927     0.833     0.878     10391
        >50K      0.612     0.801     0.694      3419

    accuracy                          0.825     13810
   macro avg      0.770     0.817     0.786     13810
weighted avg      0.849     0.825     0.832     13810

AUC: 0.907


## Model B: capital-gain and capital-loss excluded

In [69]:
data = pd.read_csv('data/adult.csv', header=None)
data.columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
data = data[['age', 'workclass', 'education-num', 'marital-status', 'occupation', 'hours-per-week', 'income']]
data = data.replace('.*\\?.*', np.nan, regex=True).dropna()
data = pd.get_dummies(data, columns=['workclass', 'marital-status', 'occupation'])

In [70]:
X = data.drop(['income'], axis=1)
y = data['income']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=21)

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

In [71]:
model = MLPClassifier(random_state=21, max_iter=1000, hidden_layer_sizes=(50,), alpha=0.05).fit(X_train, y_train)

In [72]:
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = np.where(y_prob >= 0.3, " >50K", " <=50K")

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f}")

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)
print(classification_report(y_test, y_pred, digits=3))

auc = roc_auc_score(y_test, y_prob) 
print(f"AUC: {auc:.3f}")

Accuracy: 0.804
Confusion Matrix:
 [[8403 1988]
 [ 717 2702]]
              precision    recall  f1-score   support

       <=50K      0.921     0.809     0.861     10391
        >50K      0.576     0.790     0.666      3419

    accuracy                          0.804     13810
   macro avg      0.749     0.799     0.764     13810
weighted avg      0.836     0.804     0.813     13810

AUC: 0.886


Removing investment features (`capital-gain` and `capital-loss`) resulted in only a marginal drop across all key metrics—Accuracy decreased by 2.1% (82.5% to 80.4%), Recall held steady (80.1% to 79.0%), and ROC AUC dropped slightly (0.907 to 0.886). The minimal 1.1% change in Recall proves that the model is not relying solely on investment windfalls to identify high earners. The model relies on actual earning patterns (like education and job type) rather than relying on investment shortcuts, therefore passing the leakage check.